In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr, f_oneway

In [12]:
DATA_DIR = Path("./patienten-visiten")

csv_files = sorted(DATA_DIR.glob("Patients_V*.csv"))

dfs = []

for file in csv_files:
    df = pd.read_csv(file, sep=";")
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Rows:", len(data))
print("Columns:", len(data.columns))

C:\Users\veron\AppData\Local\Temp\ipykernel_14828\3642141877.py:8: DtypeWarning: Columns (106,107,228,230,243,245,247,249,251,252,260,262,264,266,268,269,277,279,281,283,285,286,294,300,302,303,311,313,371,373,374,383,385,386,395,396,397,398,399,407,408,409,410,411,416,418,419,420,421,422,423,428,430,431,432,433,435,440,441,442,443,444,445,446,447,448,450,451,452,453,454,459,460,462,463,464,465,466,470,472,474,475,476,477,484,486,487,488,489,490,496,498,499,501,502,508,510,511,513,730,735,737,755,756,761,762,763,777,779,781,782,788,791,799,801,803,805,808,814,817,825,827,829,1016,1024,1028,1033,1034,1040,1041,1042,1046,1048,1051,1052,1058,1059,1060,1064,1066,1069,1070,1076,1077,1078,1082,1084,1087,1094,1095,1096,1099,1100,1102,1105,1112,1113,1114,1115,1117,1118,1120,1123,1124,1130,1131,1132,1133,1134,1135,1136,1138,1141,1144,1145,1148,1149,1151,1152,1153,1156,1161,1162,1163,1166,1167,1169,1170,1171,1174,1177,1178,1179,1180,1181,1184,1185,1187,1188,1189,1195,1197,1198,1199,1202,1203,120

Rows: 24497
Columns: 32084


In [13]:
score_bases = [
    "midas_result",
    "dass_depression",
    "dass_fear",
    "dass_stress",
    "vr12_msc",
    "vr12_psc",
    "gvas_result",
    "pgic_result",
    "chiq_result"
]

In [14]:
factor_prefixes = [
    "gender",
    "birthyear",
    "postal_code",
    "weight",
    "pregnant",
    "highest_degree",
    "highest_education",
    "family_status",
    "current_employed",
    "working_hours_model",
    "shiftwork",
    "working_hours_reduced",
    "promotion_waived",
    "children",
    "ch_max_birth",
    "ch_min_birth",
    "headache_days_per_month",
    "headache_days_severe_per_month",
    "days_medication",
    "intensity",
    "days_lost_work",
    "days_lost_household",
    "days_doctor",
    "days_ER",
    "days_hospital"
]


In [15]:
def to_numeric_german(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )

In [16]:
def eta_squared(groups):
    """
    Effect size for categorical variables.
    """

    groups = [g.dropna() for g in groups if len(g.dropna()) > 0]

    if len(groups) < 2:
        return np.nan

    try:
        f_oneway(*groups)
    except:
        return np.nan

    all_values = pd.concat(groups)

    grand_mean = all_values.mean()

    ss_between = sum(
        len(g) * (g.mean() - grand_mean) ** 2
        for g in groups
    )

    ss_total = ((all_values - grand_mean) ** 2).sum()

    if ss_total == 0:
        return np.nan

    return ss_between / ss_total


def detect_variable_type(series):

    s = series.dropna()

    if len(s) == 0:
        return "unknown"

    # bool
    uniques = set(map(str, s.unique()))

    if uniques <= {"0", "1", "True", "False", "true", "false"}:
        return "binary"

    # numeric?
    num = pd.to_numeric(s, errors="coerce")

    if num.notna().mean() > 0.9:

        if num.nunique() <= 2:
            return "binary"

        return "numeric"

    return "categorical"


In [17]:
factor_cols = []

for prefix in factor_prefixes:

    cols = [c for c in data.columns if c.startswith(prefix)]

    factor_cols.extend(cols)

print("Factor columns:", len(factor_cols))

Factor columns: 554


In [18]:
all_results = []

for score_base in score_bases:

    score_cols = [
        c for c in data.columns
        if c.startswith(score_base)
    ]

    if len(score_cols) == 0:
        continue

    print(f"\nProcessing {score_base}")

    for score_col in score_cols:

        score = to_numeric_german(data[score_col])

        for factor_col in factor_cols:

            tmp = pd.DataFrame({
                "factor": data[factor_col],
                "score": score
            }).dropna()

            if len(tmp) < 30:
                continue

            factor_type = detect_variable_type(
                tmp["factor"]
            )

            association = np.nan

            try:

                # ------------------------------------
                # NUMERIC
                # ------------------------------------
                if factor_type == "numeric":

                    x = pd.to_numeric(
                        tmp["factor"],
                        errors="coerce"
                    )

                    mask = x.notna()

                    if mask.sum() >= 30:

                        association, _ = spearmanr(
                            x[mask],
                            tmp.loc[mask, "score"]
                        )

                # ------------------------------------
                # BINARY
                # ------------------------------------
                elif factor_type == "binary":

                    x = pd.to_numeric(
                        tmp["factor"],
                        errors="coerce"
                    )

                    if x.nunique() == 2:

                        association = x.corr(
                            tmp["score"]
                        )

                # ------------------------------------
                # CATEGORICAL
                # ------------------------------------
                elif factor_type == "categorical":

                    groups = [
                        g["score"]
                        for _, g in tmp.groupby("factor")
                        if len(g) >= 5
                    ]

                    if len(groups) >= 2:

                        association = eta_squared(groups)

                else:
                    continue

            except:
                continue

            if pd.isna(association):
                continue

            all_results.append({
                "questionnaire": score_base,
                "score_column": score_col,
                "factor": factor_col,
                "factor_type": factor_type,
                "n": len(tmp),
                "association": association
            })


Processing midas_result

Processing dass_depression

Processing dass_fear

Processing dass_stress

Processing vr12_msc

Processing vr12_psc

Processing gvas_result

Processing pgic_result

Processing chiq_result


In [19]:
results_df = pd.DataFrame(all_results)

print("\nTotal associations:", len(results_df))


Total associations: 3775


In [20]:
for questionnaire in results_df["questionnaire"].unique():

    print("\n" + "=" * 80)
    print(questionnaire)
    print("=" * 80)

    top = (
        results_df[
            results_df["questionnaire"] == questionnaire
        ]
        .assign(
            abs_assoc=lambda d:
            d["association"].abs()
        )
        .sort_values(
            "abs_assoc",
            ascending=False
        )
        .head(20)
    )

    print(
        top[
            [
                "factor",
                "factor_type",
                "n",
                "association"
            ]
        ]
    )


midas_result
                                 factor factor_type     n  association
217             days_lost_household_K19     numeric    35     0.851549
201             days_lost_household_K18     numeric    49     0.839791
163             days_lost_household_K16     numeric    97     0.817548
415              days_lost_household_K8     numeric   564     0.784171
140             days_lost_household_K15     numeric   125     0.771901
44              days_lost_household_K11     numeric   316     0.766432
92              days_lost_household_K13     numeric   184     0.764673
68              days_lost_household_K12     numeric   251     0.764479
366              days_lost_household_K6     numeric   943     0.760703
116             days_lost_household_K14     numeric   158     0.760143
391              days_lost_household_K7     numeric   757     0.745559
184             days_lost_household_K17     numeric    68     0.743209
440              days_lost_household_K9     numeric   460     0

In [21]:
results_df.to_csv(
    "questionnaire_factor_associations.csv",
    index=False
)

print(
    "\nSaved:",
    "questionnaire_factor_associations.csv"
)


Saved: questionnaire_factor_associations.csv
